# Chess Learning — Stage 02: Self-Play & TD Learning

This notebook implements the training loop:

| Component | Description |
|---|---|
| **Self-play loop** | Engine plays both sides; positions and outcomes are recorded |
| **TD learning** | Weights are updated after each game using the outcome as a target |
| **SQLite logging** | Weight snapshots and game stats are written to a database for analysis in R |

**Design choices based on benchmarking:**
- Default depth: 2 (~3.8s/game; ~9,500 games per 10-hour run)
- Move limit: 200 half-moves per game
- Adjudication: games claimed as draws under the 50-move rule or 3-fold repetition
- Weight checkpoints every 10 games
- Per-game mean feature values logged to `feature_stats` for feature-outcome correlation analysis


## Setup

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'chess', '-q'])

0

In [2]:
import chess
import chess.pgn
import io
import sqlite3
import time
import os
from datetime import datetime

### Import engine components

In **Positron**, `engine.py` is imported directly from the same folder.
In **Google Colab**, upload `engine.py` to the session first (Files → Upload), then this import will work.


In [3]:
sys.path.insert(0, os.path.dirname(os.path.abspath('selfplay.ipynb')))
from engine import FEATURE_NAMES, default_weights, extract_features, get_best_move
print('Engine loaded. Features:', FEATURE_NAMES)

Engine loaded. Features: ['material_pawn', 'material_knight', 'material_bishop', 'material_rook', 'material_queen', 'passed_pawn', 'doubled_pawn', 'isolated_pawn', 'king_safety', 'center_control', 'rook_open_file', 'connected_rooks', 'bishop_pair', 'rook_seventh', 'piece_development', 'mobility', 'pawn_advancement']


## Database Setup

Seven tables:
- **`runs`** — one row per training run (hyperparameters, timestamp)
- **`games`** — one row per game (outcome, length, duration, termination reason)
- **`weights`** — weight snapshots every `checkpoint_every` games (primary research data)
- **`feature_stats`** — per-game mean feature values (white's perspective) for feature-outcome correlation analysis
- **`game_records`** — full PGN and mean TD error for every game; enables replay and identification of high-impact games
- **`weight_deltas`** — per-feature weight change from each game's TD update; use to find which games caused large weight jumps
- **`position_logs`** — per-ply feature vectors for every position in every game; optional (controlled by `LOG_POSITIONS`)


In [4]:
def setup_database(db_path):
    os.makedirs(os.path.dirname(os.path.abspath(db_path)), exist_ok=True)
    conn = sqlite3.connect(db_path)
    conn.executescript('''
        CREATE TABLE IF NOT EXISTS runs (
            run_id        INTEGER PRIMARY KEY,
            started_at    TEXT,
            depth         INTEGER,
            learning_rate REAL,
            move_limit    INTEGER,
            notes         TEXT
        );
        CREATE TABLE IF NOT EXISTS games (
            game_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            outcome       TEXT,
            n_halfmoves   INTEGER,
            duration_s    REAL,
            terminated_by TEXT
        );
        CREATE TABLE IF NOT EXISTS weights (
            weight_id     INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            feature_name  TEXT,
            weight_value  REAL
        );
        CREATE TABLE IF NOT EXISTS feature_stats (
            stat_id       INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            outcome       TEXT,
            feature_name  TEXT,
            mean_value    REAL
        );
        CREATE TABLE IF NOT EXISTS game_records (
            record_id     INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            pgn           TEXT,
            td_error      REAL
        );
        CREATE TABLE IF NOT EXISTS weight_deltas (
            delta_id      INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            feature_name  TEXT,
            delta         REAL
        );
        CREATE TABLE IF NOT EXISTS position_logs (
            pos_id        INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            ply           INTEGER,
            color         TEXT,
            feature_name  TEXT,
            feature_value REAL
        );
    ''')
    conn.commit()
    # Safe schema migration: add last_updated column if not already present
    try:
        conn.execute('ALTER TABLE runs ADD COLUMN last_updated TEXT')
        conn.commit()
    except sqlite3.OperationalError:
        pass  # column already exists
    return conn


def log_run(conn, run_id, depth, learning_rate, move_limit, notes=''):
    conn.execute(
        'INSERT OR REPLACE INTO runs (run_id, started_at, depth, learning_rate, move_limit, notes) '
        'VALUES (?, ?, ?, ?, ?, ?)',
        (run_id, datetime.now().isoformat(), depth, learning_rate, move_limit, notes)
    )
    conn.commit()


def log_game(conn, run_id, game_number, outcome, n_halfmoves, duration_s, terminated_by):
    conn.execute(
        'INSERT INTO games (run_id,game_number,outcome,n_halfmoves,duration_s,terminated_by) '
        'VALUES (?,?,?,?,?,?)',
        (run_id, game_number, outcome, n_halfmoves, duration_s, terminated_by)
    )
    conn.commit()


def log_weights(conn, run_id, game_number, weights):
    rows = [(run_id, game_number, name, val) for name, val in weights.items()]
    conn.executemany(
        'INSERT INTO weights (run_id,game_number,feature_name,weight_value) VALUES (?,?,?,?)',
        rows
    )
    conn.commit()


def log_feature_stats(conn, run_id, game_number, positions, outcome):
    # Compute mean of each feature across all positions, converted to white's perspective.
    # White-turn positions: use value as-is (positive = good for white).
    # Black-turn positions: negate (feature is from black's perspective, flip for white).
    # This puts all games on a common scale for feature-outcome correlation analysis.
    if not positions:
        return
    sums = {name: 0.0 for name in FEATURE_NAMES}
    for features, color in positions:
        sign = 1.0 if color == chess.WHITE else -1.0
        for name, val in features.items():
            sums[name] += sign * val
    n = len(positions)
    rows = [(run_id, game_number, outcome, name, sums[name] / n) for name in FEATURE_NAMES]
    conn.executemany(
        'INSERT INTO feature_stats (run_id,game_number,outcome,feature_name,mean_value) VALUES (?,?,?,?,?)',
        rows
    )
    conn.commit()


def load_last_weights(conn, run_id):
    # Load weights from the most recent checkpoint for a given run_id.
    max_game = conn.execute(
        'SELECT MAX(game_number) FROM weights WHERE run_id = ?', (run_id,)
    ).fetchone()[0]
    if max_game is None:
        raise ValueError(f'No weight checkpoints found for run_id={run_id}')
    rows = conn.execute(
        'SELECT feature_name, weight_value FROM weights '
        'WHERE run_id = ? AND game_number = ?',
        (run_id, max_game)
    ).fetchall()
    return {name: val for name, val in rows}


def get_last_game_number(conn, run_id):
    # Return the highest game_number logged for a given run_id (0 if none).
    result = conn.execute(
        'SELECT MAX(game_number) FROM games WHERE run_id = ?', (run_id,)
    ).fetchone()[0]
    return result if result is not None else 0


def log_game_record(conn, run_id, game_number, pgn, td_error):
    conn.execute(
        'INSERT INTO game_records (run_id,game_number,pgn,td_error) VALUES (?,?,?,?)',
        (run_id, game_number, pgn, td_error)
    )
    conn.commit()


def log_weight_deltas(conn, run_id, game_number, deltas):
    rows = [(run_id, game_number, name, delta) for name, delta in deltas.items()]
    conn.executemany(
        'INSERT INTO weight_deltas (run_id,game_number,feature_name,delta) VALUES (?,?,?,?)',
        rows
    )
    conn.commit()


def log_position_logs(conn, run_id, game_number, positions):
    rows = []
    for ply, (features, color) in enumerate(positions):
        color_str = 'white' if color == chess.WHITE else 'black'
        for name, val in features.items():
            rows.append((run_id, game_number, ply, color_str, name, val))
    conn.executemany(
        'INSERT INTO position_logs (run_id,game_number,ply,color,feature_name,feature_value) '
        'VALUES (?,?,?,?,?,?)',
        rows
    )
    conn.commit()

## Self-Play Game Function

Each call plays one complete game. For each half-move, the board features and side-to-move are recorded so the TD update can process every position.

Termination conditions (in priority order):
1. Natural game end (checkmate, stalemate, insufficient material)
2. Claim draw: 50-move rule or 3-fold repetition (`board.can_claim_draw()`)
3. Hard move limit (safety net for very long games with near-zero weights)


In [5]:
def play_game(weights, depth, move_limit=200):
    board = chess.Board()
    game  = chess.pgn.Game()
    node  = game
    positions     = []       # list of (features_dict, color_to_move)
    terminated_by = 'natural'

    while True:
        if board.is_game_over():
            terminated_by = 'natural'
            break
        if board.can_claim_draw():
            terminated_by = 'draw_claim'
            break
        if len(positions) >= move_limit:
            terminated_by = 'move_limit'
            break

        features = extract_features(board)
        color    = board.turn

        move, _ = get_best_move(board, weights, depth=depth, randomize=True)
        if move is None:
            break

        node = node.add_variation(move)
        board.push(move)
        positions.append((features, color))

    # Resolve outcome
    if terminated_by in ('draw_claim', 'move_limit'):
        outcome = '1/2-1/2'
    else:
        outcome = board.result()

    game.headers['Result'] = outcome
    pgn_str = str(game)

    return positions, outcome, terminated_by, pgn_str

## TD Learning

After each game, every recorded position is used to update the weights. The target is the game outcome from the perspective of the side to move at that position. The update rule is:

$$w_i \leftarrow w_i + \alpha \cdot (\text{target} - \text{prediction}) \cdot x_i$$

where $$\alpha$$ is the learning rate, $$\text{prediction} = \mathbf{w} \cdot \mathbf{x}$$, and $$x_i$$ is the feature value. This is gradient descent on squared prediction error.

Targets:
- Side to move wins → `+1`
- Side to move loses → `-1`
- Draw → `0`


In [6]:
def outcome_for_color(outcome, color):
    if outcome == '1-0':
        return  1.0 if color == chess.WHITE else -1.0
    elif outcome == '0-1':
        return -1.0 if color == chess.WHITE else  1.0
    return 0.0  # draw or unknown


def td_update(weights, positions, outcome, learning_rate):
    weights_before = dict(weights)
    total_error    = 0.0
    for features, color in positions:
        target     = outcome_for_color(outcome, color)
        prediction = sum(weights.get(name, 0.0) * val for name, val in features.items())
        error      = target - prediction
        total_error += abs(error)
        for name, val in features.items():
            weights[name] += learning_rate * error * val
    # Clip weights to prevent blow-up
    for name in weights:
        weights[name] = max(-50.0, min(50.0, weights[name]))
    deltas   = {name: weights[name] - weights_before[name] for name in weights}
    td_error = total_error / len(positions) if positions else 0.0
    return deltas, td_error

## Training Loop

The training loop:
1. Plays a game
2. Updates weights via TD learning
3. Logs the game result and stats
4. Checkpoints weights to SQLite every `checkpoint_every` games

**Resume mode** (`resume=True`): loads the last saved weights for `run_id` and continues game numbering from where the previous session left off. All games remain under the same `run_id`, so the weight trajectory is a single continuous series in the database.

Weight checkpoints at game 0 (initial) and at every checkpoint interval are what the R visualization notebook reads.

In [7]:
def train(
    run_id,
    n_games,
    depth            = 2,
    learning_rate    = 0.001,
    move_limit       = 200,
    checkpoint_every = 10,
    db_path          = '../../data/chess_learning.db',
    notes            = '',
    resume           = False,
    log_positions    = True,
    verbose          = True,
):
    conn = setup_database(db_path)

    if resume:
        weights    = load_last_weights(conn, run_id)
        start_game = get_last_game_number(conn, run_id) + 1
        conn.execute(
            'UPDATE runs SET last_updated = ? WHERE run_id = ?',
            (datetime.now().isoformat(), run_id)
        )
        conn.commit()
    else:
        weights    = default_weights('random')
        start_game = 1
        log_run(conn, run_id, depth, learning_rate, move_limit, notes)
        log_weights(conn, run_id, 0, weights)  # checkpoint initial weights

    outcomes = {'1-0': 0, '0-1': 0, '1/2-1/2': 0}
    t_start  = time.time()

    for i in range(n_games):
        game_num = start_game + i
        t0       = time.time()
        positions, outcome, terminated_by, pgn = play_game(weights, depth, move_limit)
        elapsed  = time.time() - t0

        deltas, td_error = td_update(weights, positions, outcome, learning_rate)
        log_game(conn, run_id, game_num, outcome, len(positions), elapsed, terminated_by)
        log_game_record(conn, run_id, game_num, pgn, td_error)
        log_weight_deltas(conn, run_id, game_num, deltas)
        log_feature_stats(conn, run_id, game_num, positions, outcome)
        if log_positions:
            log_position_logs(conn, run_id, game_num, positions)
        outcomes[outcome] = outcomes.get(outcome, 0) + 1

        if game_num % checkpoint_every == 0:
            log_weights(conn, run_id, game_num, weights)
            if verbose:
                elapsed_total = time.time() - t_start
                rate = (i + 1) / elapsed_total * 3600
                print(
                    f'Game {game_num:5d} | {outcome:7s} | {len(positions):3d} half-moves '
                    f'| {elapsed:.1f}s | td_err={td_error:.3f} '
                    f'| {rate:.0f} games/hr'
                )

    final_game = start_game + n_games - 1
    log_weights(conn, run_id, final_game, weights)  # final checkpoint
    conn.close()

    total_time = time.time() - t_start
    if verbose:
        action = 'resumed' if resume else 'complete'
        print(f'\nRun {run_id} {action}: {n_games} games in {total_time/60:.1f} min '
              f'(games {start_game} to {final_game})')
        print(f'Outcomes: {outcomes}')
        print(f'Database: {os.path.abspath(db_path)}')

    return weights

## Configuration & Run

Edit the values below to configure a training session. Each run gets a unique `RUN_ID`; multiple sessions can contribute to the same run by setting `RESUME = True`.

**Typical session sizes:**
- Quick check / first session: `N_GAMES = 100` (~15–20 min)
- Medium session: `N_GAMES = 500` (~60–90 min)

**Workflow:**
1. Set `RESUME = False` and run to start a new run.
2. Examine results in R (`03_visualize.qmd`).
3. Set `RESUME = True` and run again with the same `RUN_ID` to continue.

> **Google Colab path**: set `DB_PATH` to a location on your mounted Drive,  
> e.g. `'/content/drive/MyDrive/chess_learning.db'`


In [8]:
RUN_ID           = 2
N_GAMES          = 500       # games to play in this session
DEPTH            = 2
LEARNING_RATE    = 0.001
MOVE_LIMIT       = 200       # half-moves; safety cap for early training
CHECKPOINT_EVERY = 10        # log weights to DB every N games
DB_PATH          = '../../data/chess_learning.db'
RESUME           = True      # set True to continue an existing run from where it left off
LOG_POSITIONS    = True      # log per-ply feature vectors; disable to save space on long runs

final_weights = train(
    run_id           = RUN_ID,
    n_games          = N_GAMES,
    depth            = DEPTH,
    learning_rate    = LEARNING_RATE,
    move_limit       = MOVE_LIMIT,
    checkpoint_every = CHECKPOINT_EVERY,
    db_path          = DB_PATH,
    resume           = RESUME,
    log_positions    = LOG_POSITIONS,
    notes            = f'depth={DEPTH}, lr={LEARNING_RATE}',
)

Game   260 | 1/2-1/2 |  40 half-moves | 2.4s | td_err=0.007 | 1731 games/hr


Game   270 | 1/2-1/2 |  25 half-moves | 1.3s | td_err=0.007 | 1787 games/hr


Game   280 | 1/2-1/2 |  36 half-moves | 2.0s | td_err=0.005 | 1739 games/hr


Game   290 | 1/2-1/2 |  31 half-moves | 1.7s | td_err=0.006 | 1767 games/hr


Game   300 | 1/2-1/2 |  42 half-moves | 2.4s | td_err=0.005 | 1741 games/hr


...

Game   710 | 1/2-1/2 |  60 half-moves | 3.6s | td_err=0.000 | 1829 games/hr


Game   720 | 1/2-1/2 |  34 half-moves | 2.1s | td_err=0.001 | 1826 games/hr


Game   730 | 1/2-1/2 |  25 half-moves | 1.3s | td_err=0.001 | 1822 games/hr


Game   740 | 1/2-1/2 |  34 half-moves | 2.2s | td_err=0.000 | 1823 games/hr


Game   750 | 1/2-1/2 |  25 half-moves | 1.2s | td_err=0.000 | 1824 games/hr

Run 2 resumed: 500 games in 16.4 min (games 251 to 750)
Outcomes: {'1-0': 0, '0-1': 0, '1/2-1/2': 500}
Database: C:\Users\dkill\OneDrive\Documents\Chess Learning\data\chess_learning.db


## Weight Trajectory

Quick check: read weight snapshots back from the database and display how each feature weight evolved over the run. Richer visualization will be built in Stage 03 (R).


In [9]:
conn = sqlite3.connect(DB_PATH)
rows = conn.execute(
    'SELECT game_number, feature_name, weight_value FROM weights '
    'WHERE run_id = ? ORDER BY game_number, feature_name',
    (RUN_ID,)
).fetchall()
conn.close()

# Pivot to game_number -> feature -> value
from collections import defaultdict
by_game = defaultdict(dict)
for game_num, feature, val in rows:
    by_game[game_num][feature] = val

game_nums = sorted(by_game.keys())

header = f"{'Feature':<20}" + ''.join(f'{g:>10}' for g in game_nums)
print(header)
print('-' * len(header))
for feature in FEATURE_NAMES:
    row = f'{feature:<20}' + ''.join(f'{by_game[g].get(feature, 0):>10.4f}' for g in game_nums)
    print(row)

Feature                      0        10        20        30        40        50        60        70        80        90       100       110       120       130       140       150       160       170       180       190       200       210       220       230       240       250       260       270       280       290       300       310       320       330       340       350       360       370       380       390       400       410       420       430       440       450       460       470       480       490       500       510       520       530       540       550       560       570       580       590       600       610       620       630       640       650       660       670       680       690       700       710       720       730       740       750
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

---
If weights are moving (even slightly), the pipeline is working. Pronounced movement in `material_*` weights is expected first — this is the engine beginning to value pieces correctly. Weight trajectories will be visualized fully in Stage 03 (R).